# Interactive Image Processing Application

Welcome to the Interactive Image Processing Lab. In this lab, you will build an interactive image processing application using OpenCV, Matplotlib, and ipywidgets. You will implement various functionalities such as image browsing, grayscale conversion, brightness adjustment, resizing, rotation, and histogram plotting.

Follow the steps below to complete the lab.

In [ ]:
# Import necessary libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import widgets
from IPython.display import display
import os

# Global variables for storing image data
original_bgr, original_rgb, processed_rgb = None, None, None

# Output areas for status messages, images, and histogram
msg_out = widgets.Output()
img_out = widgets.Output()
hist_out = widgets.Output()

In [ ]:
# Helper function to display images
def show_images(original_rgb, processed_rgb):
    with img_out:
        img_out.clear_output()
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        ax[0].imshow(original_rgb)
        ax[0].set_title('Original Image')
        ax[0].axis('off')
        ax[1].imshow(processed_rgb)
        ax[1].set_title('Processed Image')
        ax[1].axis('off')
        plt.show()

# Helper function to plot histogram
def show_histogram(processed_rgb):
    with hist_out:
        hist_out.clear_output()
        gray = cv2.cvtColor(processed_rgb, cv2.COLOR_RGB2GRAY)
        plt.figure(figsize=(5, 3))
        plt.hist(gray.ravel(), 256, [0, 256])
        plt.title('Histogram')
        plt.show()

In [ ]:
# UI controls
browse_btn = widgets.Button(description='Browse Image')
grayscale_btn = widgets.Button(description='Convert to Grayscale')
brightness_slider = widgets.IntSlider(description='Brightness', min=-100, max=100)
resize_dropdown = widgets.Dropdown(description='Resize', options=[1.0, 0.75, 0.5])
rotate_dropdown = widgets.Dropdown(description='Rotation', options=[0, 90, 180, 270])
histogram_btn = widgets.Button(description='Plot Histogram')
save_btn = widgets.Button(description='Save Image')

In [ ]:
# Callback for Browse Image
def on_browse_image(change):
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        global original_bgr, original_rgb, processed_rgb
        original_bgr = cv2.imdecode(np.frombuffer(uploaded[fn], np.uint8), cv2.IMREAD_COLOR)
        if original_bgr is not None:
            original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
            processed_rgb = original_rgb.copy()
            show_images(original_rgb, processed_rgb)
            with msg_out:
                msg_out.clear_output()
                print(f'Loaded {fn}')

# Callback for Convert to Grayscale
def on_convert_to_grayscale(change):
    global processed_rgb
    gray = cv2.cvtColor(original_rgb, cv2.COLOR_RGB2GRAY)
    processed_rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    show_images(original_rgb, processed_rgb)

# Callback for Brightness Adjustment
def on_brightness_change(change):
    global processed_rgb
    brightness = change['new']
    adjusted = np.clip(original_rgb + brightness, 0, 255).astype(np.uint8)
    processed_rgb = adjusted
    show_images(original_rgb, processed_rgb)

# Callback for Resize
def on_resize(change):
    global processed_rgb
    scale = change['new']
    width = int(original_rgb.shape[1] * scale)
    height = int(original_rgb.shape[0] * scale)
    processed_rgb = cv2.resize(original_rgb, (width, height), interpolation=cv2.INTER_AREA)
    show_images(original_rgb, processed_rgb)
    with msg_out:
        msg_out.clear_output()
        print(f'Resized to {processed_rgb.shape[1]}x{processed_rgb.shape[0]}')

# Callback for Rotation
def on_rotate(change):
    global processed_rgb
    angle = change['new']
    if angle == 90:
        processed_rgb = cv2.rotate(original_rgb, cv2.ROTATE_90_CLOCKWISE)
    elif angle == 180:
        processed_rgb = cv2.rotate(original_rgb, cv2.ROTATE_180)
    elif angle == 270:
        processed_rgb = cv2.rotate(original_rgb, cv2.ROTATE_90_COUNTERCLOCKWISE)
    else:
        processed_rgb = original_rgb.copy()
    show_images(original_rgb, processed_rgb)

# Callback for Plot Histogram
def on_plot_histogram(change):
    show_histogram(processed_rgb)

# Callback for Save Image
def on_save_image(change):
    global processed_rgb
    output_path = './outputs/'
    os.makedirs(output_path, exist_ok=True)
    filename = os.path.join(output_path, 'processed_image.png')
    cv2.imwrite(filename, cv2.cvtColor(processed_rgb, cv2.COLOR_RGB2BGR))
    with msg_out:
        msg_out.clear_output()
        print(f'Saved image to {filename}')

In [ ]:
# Connect callbacks
browse_btn.on_click(on_browse_image)
grayscale_btn.on_click(on_convert_to_grayscale)
brightness_slider.observe(on_brightness_change, names='value')
resize_dropdown.observe(on_resize, names='value')
rotate_dropdown.observe(on_rotate, names='value')
histogram_btn.on_click(on_plot_histogram)
save_btn.on_click(on_save_image)

# Layout
controls = widgets.VBox([browse_btn, grayscale_btn, brightness_slider, resize_dropdown, rotate_dropdown, histogram_btn, save_btn])
outputs = widgets.VBox([img_out, hist_out])
layout = widgets.HBox([controls, outputs])

display(layout)
msg_out.append_stdout('Please click Browse Image to start.\n')